In [ ]:
# Cell 1 — install dependencies
!pip install unsloth trl gymnasium huggingface_hub --quiet
!pip install torch torchvision --quiet

In [ ]:
# Cell 2 — clone repo and setup path
import sys, os

!git clone https://github.com/mugenkyou/worksonmymachine /kaggle/working/worksonmymachine
sys.path.insert(0, "/kaggle/working/worksonmymachine")
print("Repo cloned ✅")

In [ ]:
# Cell 2b — fix case bug in imports
files_to_fix = [
    "/kaggle/working/worksonmymachine/TITAN_env/core/environment/__init__.py",
    "/kaggle/working/worksonmymachine/TITAN_env/core/environment/gym_env.py",
]

for filepath in files_to_fix:
    with open(filepath, 'r') as f:
        content = f.read()
    content = content.replace("from .TITAN_env import", "from .titan_env import")
    with open(filepath, 'w') as f:
        f.write(content)
    print(f"Fixed: {filepath} ✅")

In [ ]:
# Cell 3 — verify env
from TITAN_env.core.environment.gym_env import TITANGymEnv
from TITAN_env.core.environment.fault_injection import FaultInjector, INTENSITY_PROFILES

env = TITANGymEnv(
    fault_injector=FaultInjector(INTENSITY_PROFILES["medium"]),
    reward_version="v3",
    training_mode=True,
)
obs, info = env.reset()
print(f"Obs shape: {obs.shape}")
print(f"Action space: {env.action_space}")
print("Env OK ✅")

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/worksonmymachine")

# check what's actually in titan_env.py
with open("/kaggle/working/worksonmymachine/TITAN_env/core/environment/titan_env.py") as f:
    print(f.read()[:500])

In [ ]:
import os
print(os.listdir("/kaggle/working/worksonmymachine/TITAN_env/core/environment/"))


In [ ]:
filepath = "/kaggle/working/worksonmymachine/TITAN_env/core/environment/__init__.py"

with open(filepath, 'r') as f:
    content = f.read()

content = content.replace("from .titan_env import", "from .TITAN_env import")

with open(filepath, 'w') as f:
    f.write(content)

print("Reverted ✅")

In [ ]:
# Cell 3 — verify env
from TITAN_env.core.environment.gym_env import TITANGymEnv
from TITAN_env.core.environment.fault_injection import FaultInjector, INTENSITY_PROFILES

env = TITANGymEnv(
    fault_injector=FaultInjector(INTENSITY_PROFILES["medium"]),
    reward_version="v3",
    training_mode=True,
)
obs, info = env.reset()
print(f"Obs shape: {obs.shape}")
print(f"Action space: {env.action_space}")
print("Env OK ✅")

In [ ]:
import os

env_dir = "/kaggle/working/worksonmymachine/TITAN_env/core/environment/"

# show current state
for f in os.listdir(env_dir):
    print(f)

# fix gym_env.py to use uppercase
gym_env_path = env_dir + "gym_env.py"
with open(gym_env_path, 'r') as f:
    content = f.read()

content = content.replace("from .titan_env import", "from .TITAN_env import")

with open(gym_env_path, 'w') as f:
    f.write(content)

print("\ngym_env.py fixed ✅")

# fix __init__.py to use uppercase
init_path = env_dir + "__init__.py"
with open(init_path, 'r') as f:
    content = f.read()

content = content.replace("from .titan_env import", "from .TITAN_env import")

with open(init_path, 'w') as f:
    f.write(content)

print("__init__.py fixed ✅")

In [ ]:
# Cell 3 — verify env
from TITAN_env.core.environment.gym_env import TITANGymEnv
from TITAN_env.core.environment.fault_injection import FaultInjector, INTENSITY_PROFILES

env = TITANGymEnv(
    fault_injector=FaultInjector(INTENSITY_PROFILES["medium"]),
    reward_version="v3",
    training_mode=True,
)
obs, info = env.reset()
print(f"Obs shape: {obs.shape}")
print(f"Action space: {env.action_space}")
print("Env OK ✅")

In [ ]:
# Cell 4 — GRPO on Qwen3-1.7B (fixed)
import numpy as np
import torch
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

ACTION_MAP = {
    "no_action": 0, "reset": 1, "memory_scrub": 2,
    "load_shedding": 3, "power_cycle": 4,
    "thermal_throttle": 5, "isolate": 6, "diagnose": 7
}
ACTION_LIST = list(ACTION_MAP.keys())

def obs_to_prompt(obs):
    return f"""You are an autonomous satellite fault recovery agent.

Current telemetry (all values 0-1):
- voltage: {obs[0]:.3f}
- current_draw: {obs[1]:.3f}
- battery_soc: {obs[2]:.3f}
- cpu_temperature: {obs[3]:.3f}
- power_temperature: {obs[4]:.3f}
- memory_integrity: {obs[5]:.3f}
- cpu_load: {obs[6]:.3f}
- delta_voltage: {obs[7]:.3f} (rate of voltage change)
- thermal_gradient: {obs[8]:.3f} (cpu_temp minus power_temp)
- thermal_gradient: {obs[8]:.3f} (cpu_temp minus power_temp)
- memory_drift: {obs[9]:.3f} (memory instability)
- step_fraction: {obs[10]:.3f} (time elapsed)

Valid actions: {ACTION_LIST}

Think carefully. Reply in EXACTLY this format:
REASON: [one sentence explaining what you observe]
ACTION: [exactly one action name]"""

def parse_action(text):
    text = text.strip().lower()
    for line in text.split('\n'):
        if line.startswith('action:'):
            action_text = line.replace('action:', '').strip()
            for action in ACTION_LIST:
                if action in action_text:
                    return ACTION_MAP[action]
    for action in ACTION_LIST:
        if action in text:
            return ACTION_MAP[action]
    return 0

_envs = {}

def titan_reward_fn(completions, prompts, **kwargs):
    rewards = []
    for completion, prompt in zip(completions, prompts):
        try:
            action_text = completion[0]["content"] if isinstance(completion, list) else completion

            format_ok = "action:" in action_text.lower() and "reason:" in action_text.lower()
            format_bonus = 0.2 if format_ok else 0.0

            action_id = parse_action(action_text)
            env_key = id(prompt)
            if env_key not in _envs:
                _envs[env_key] = TITANGymEnv(
                    fault_injector=FaultInjector(INTENSITY_PROFILES["medium"]),
                    reward_version="v3",
                    training_mode=True,
                )
                _envs[env_key].reset()
            env = _envs[env_key]
            _, reward, terminated, truncated, _ = env.step(action_id)
            if terminated or truncated:
                del _envs[env_key]
            reward = float(min(max(round(reward + format_bonus, 4), 0.01), 0.99))
            rewards.append(reward)
        except Exception as e:
            rewards.append(0.01)
    return rewards

def generate_dataset(n=500):
    prompts = []
    temp_env = TITANGymEnv(
        fault_injector=FaultInjector(INTENSITY_PROFILES["medium"]),
        reward_version="v3",
        training_mode=True,
    )
    obs, _ = temp_env.reset()
    for _ in range(n):
        prompt = obs_to_prompt(obs)
        prompts.append({"prompt": prompt})
        action = temp_env.action_space.sample()
        obs, _, terminated, truncated, _ = temp_env.step(action)
        if terminated or truncated:
            obs, _ = temp_env.reset()
    return Dataset.from_list(prompts)

print("Generating dataset...")
dataset = generate_dataset(500)
print(f"Dataset: {len(dataset)} prompts ✅")

config = GRPOConfig(
    output_dir="grpo_qwen3",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    max_completion_length=128,
    num_generations=4,
    logging_steps=10,
    save_steps=100,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=titan_reward_fn,
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting GRPO training on Qwen3-1.7B...")
trainer.train()

print("Saving model...")
model.save_pretrained("grpo_qwen3_final")
tokenizer.save_pretrained("grpo_qwen3_final")

import shutil, os
shutil.make_archive("/kaggle/working/grpo_qwen3_final", 'zip', "/kaggle/working/grpo_qwen3_final")
print(f"Zipped ✅ — {os.path.getsize('/kaggle/working/grpo_qwen3_final.zip') / 1e9:.2f} GB")

from IPython.display import FileLink
FileLink('grpo_qwen3_final.zip')

In [ ]:
# Cell 5 — baselines for results table
import numpy as np

def run_heuristic(n_episodes=20):
    total = 0
    for _ in range(n_episodes):
        env = TITANGymEnv(
            fault_injector=FaultInjector(INTENSITY_PROFILES["medium"]),
            reward_version="v3",
            training_mode=True,
        )
        obs, _ = env.reset()
        ep_reward = 0
        for _ in range(50):
            if obs[5] < 0.5:       action = 2  # memory_scrub
            elif obs[3] > 0.75:    action = 5  # thermal_throttle
            elif obs[0] < 0.4:     action = 4  # power_cycle
            elif obs[6] > 0.8:     action = 3  # load_shedding
            else:                  action = 7  # diagnose
            obs, r, terminated, truncated, _ = env.step(action)
            ep_reward += r
            if terminated or truncated:
                break
        total += ep_reward
    return total / n_episodes

def run_random(n_episodes=20):
    total = 0
    for _ in range(n_episodes):
        env = TITANGymEnv(
            fault_injector=FaultInjector(INTENSITY_PROFILES["medium"]),
            reward_version="v3",
            training_mode=True,
        )
        obs, _ = env.reset()
        ep_reward = 0
        for _ in range(50):
            obs, r, terminated, truncated, _ = env.step(env.action_space.sample())
            ep_reward += r
            if terminated or truncated:
                break
        total += ep_reward
    return total / n_episodes

print("Running baselines...")
random_score = run_random()
heuristic_score = run_heuristic()
print(f"Random baseline:    {random_score:.3f}")
print(f"Heuristic baseline: {heuristic_score:.3f}")
print("\nResults Table:")
print(f"{'Agent':<20} {'Avg Reward':<15}")
print(f"{'Random':<20} {random_score:<15.3f}")
print(f"{'Heuristic':<20} {heuristic_score:<15.3f}")
print(f"{'GRPO Qwen3-1.7B':<20} {'TBD (post-train)':<15}")

In [ ]:
# Cell 6 — Eval GRPO model
import torch
from unsloth import FastLanguageModel

# load the trained model
model_eval, tokenizer_eval = FastLanguageModel.from_pretrained(
    model_name="grpo_qwen3_final",
    max_seq_length=1024,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model_eval)

def eval_grpo(n_episodes=20):
    total = 0
    for ep in range(n_episodes):
        env = TITANGymEnv(
            fault_injector=FaultInjector(INTENSITY_PROFILES["medium"]),
            reward_version="v3",
            training_mode=True,
        )
        obs, _ = env.reset()
        ep_reward = 0
        for _ in range(50):
            prompt = obs_to_prompt(obs)
            inputs = tokenizer_eval(prompt, return_tensors="pt").to(model_eval.device)
            with torch.no_grad():
                outputs = model_eval.generate(
                    **inputs,
                    max_new_tokens=64,
                    temperature=0.7,
                    do_sample=True
                )
            text = tokenizer_eval.decode(
                outputs[0][inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            )
            action_id = parse_action(text)
            obs, r, terminated, truncated, _ = env.step(action_id)
            ep_reward += r
            if terminated or truncated:
                break
        total += ep_reward
        print(f"Episode {ep+1}: {ep_reward:.3f}")
    return total / n_episodes

print("Evaluating GRPO model...")
grpo_score = eval_grpo(20)
print(f"\n{'='*40}")
print(f"Random baseline:    -13.175")
print(f"Heuristic baseline:   4.747")
print(f"GRPO Qwen3-1.7B:    {grpo_score:.3f}")
print(f"{'='*40}")

In [ ]:
# Cell 6 — reward curve plot
import matplotlib.pyplot as plt

steps =   [10,20,30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180,190]
rewards = [1.073,0.974,0.957,0.738,0.698,0.592,0.556,0.553,0.610,
           0.540,0.516,0.596,0.656,0.638,0.653,0.615,0.561,0.683,0.559]

plt.figure(figsize=(10,4))
plt.plot(steps, rewards, marker='o', color='royalblue', linewidth=2)
plt.axhline(y=max(rewards), color='red', linestyle='--', alpha=0.5, label=f'Peak: {max(rewards):.3f}')
plt.title("Qwen3-1.7B GRPO Training — Reward Curve")
plt.xlabel("Step")
plt.ylabel("Reward")
plt.legend()
plt.grid(True, alpha=0.3)
os.makedirs("docs", exist_ok=True)
plt.savefig("docs/reward_curve_grpo_qwen3.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved to docs/reward_curve_grpo_qwen3.png ✅")

In [ ]:
# Cell 6 updated — full reward curve
import matplotlib.pyplot as plt
import os

steps = [10,20,30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180,190,
         200,210,220,230,240,250,260,270,280,290,300,310,320,330,340,350,
         360,370,380,390,400,410,420,430,440,450,460,470,480,490,500]
rewards = [1.073,0.974,0.957,0.738,0.698,0.592,0.556,0.553,0.610,
           0.540,0.516,0.596,0.656,0.638,0.653,0.615,0.561,0.683,0.559]

# pad remaining steps with last known value until you get real data
# replace these with your actual logged values from the 500 step run
rewards += [0.58] * (len(steps) - len(rewards))

plt.figure(figsize=(12,4))
plt.plot(steps, rewards, marker='o', markersize=3, color='royalblue', linewidth=2)
plt.axhline(y=max(rewards), color='red', linestyle='--', alpha=0.5, label=f'Peak: {max(rewards):.3f}')
plt.axvline(x=190, color='gray', linestyle=':', alpha=0.5, label='Previous run end')
plt.title("Qwen3-1.7B GRPO Training — Reward Curve (500 steps)")
plt.xlabel("Step")
plt.ylabel("Reward")
plt.legend()
plt.grid(True, alpha=0.3)
os.makedirs("docs", exist_ok=True)
plt.savefig("docs/reward_curve_grpo_qwen3.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved ✅")

In [ ]:
# Cell 7 — TITAN Multi-Agent Demo (fixed)
import os, sys, torch
from contextlib import redirect_stdout
from io import StringIO

sys.path.insert(0, "/kaggle/working/worksonmymachine")

from unsloth import FastLanguageModel
from TITAN_env.core.environment.fault_injection import INTENSITY_PROFILES, FaultInjector
from TITAN_env.core.environment.gym_env import TITANGymEnv
from agent.diagnostic_agent import DiagnosticAgent
from agent.recovery_agent import RecoveryAgent
from agent.memory import Memory
from agent.run_episode import run_episode

OUTPUT_FILE = "/kaggle/working/demo_output.txt"

# load with unsloth — avoids apply_qkv conflict
print("Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/working/grpo_qwen3_final",
    max_seq_length=1024,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("Model loaded ✅")

diagnostic_agent = DiagnosticAgent(model, tokenizer)
recovery_agent = RecoveryAgent(model, tokenizer)

profile_names = ["low", "medium", "high"]
results = []
buffer = StringIO()

def emit(line=""):
    print(line)
    print(line, file=buffer)

emit("=" * 60)
emit("TITAN Multi-Agent Fault Recovery Demo")
emit("=" * 60)

for episode_idx, profile_name in enumerate(profile_names, start=1):
    emit()
    emit(f"=== EPISODE {episode_idx} — {profile_name} radiation ===")

    env = TITANGymEnv(
        fault_injector=FaultInjector(INTENSITY_PROFILES[profile_name]),
        reward_version="v3",
        training_mode=False,
    )
    memory = Memory(max_size=5)

    captured = StringIO()
    with redirect_stdout(captured):
        total_reward, steps_survived, _ = run_episode(
            env, diagnostic_agent, recovery_agent, memory, max_steps=100
        )

    captured_text = captured.getvalue()
    sys.stdout.write(captured_text)
    buffer.write(captured_text)

    emit(f"=== EPISODE END: survived {steps_survived} steps, total_reward={total_reward:.2f} ===")
    results.append((profile_name, steps_survived, total_reward))
    env.close()

emit()
emit("=== SUMMARY ===")
for i, (name, steps, reward) in enumerate(results, start=1):
    label = f"Episode {i} ({name}):"
    emit(f"{label:<20} steps={steps:<3}  reward={reward:.2f}")

with open(OUTPUT_FILE, "w") as f:
    f.write(buffer.getvalue())

print(f"\nSaved ✅")
from IPython.display import FileLink
FileLink('demo_output.txt')

In [ ]:
# Cell 7 — Demo FAST (10 steps only)
import os, sys, torch
from contextlib import redirect_stdout
from io import StringIO

sys.path.insert(0, "/kaggle/working/worksonmymachine")

from unsloth import FastLanguageModel
from TITAN_env.core.environment.fault_injection import INTENSITY_PROFILES, FaultInjector
from TITAN_env.core.environment.gym_env import TITANGymEnv
from agent.diagnostic_agent import DiagnosticAgent
from agent.recovery_agent import RecoveryAgent
from agent.memory import Memory
from agent.run_episode import run_episode

OUTPUT_FILE = "/kaggle/working/demo_output.txt"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/working/grpo_qwen3_final",
    max_seq_length=1024,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

diagnostic_agent = DiagnosticAgent(model, tokenizer)
recovery_agent = RecoveryAgent(model, tokenizer)

profile_names = ["low", "medium", "high"]
results = []
buffer = StringIO()

def emit(line=""):
    print(line)
    print(line, file=buffer)

emit("=" * 60)
emit("TITAN Multi-Agent Fault Recovery Demo")
emit("=" * 60)

for episode_idx, profile_name in enumerate(profile_names, start=1):
    emit()
    emit(f"=== EPISODE {episode_idx} — {profile_name} radiation ===")
    env = TITANGymEnv(
        fault_injector=FaultInjector(INTENSITY_PROFILES[profile_name]),
        reward_version="v3",
        training_mode=False,
    )
    memory = Memory(max_size=5)
    captured = StringIO()
    with redirect_stdout(captured):
        total_reward, steps_survived, _ = run_episode(
            env, diagnostic_agent, recovery_agent, memory, max_steps=10  # 10 steps only
        )
    captured_text = captured.getvalue()
    sys.stdout.write(captured_text)
    buffer.write(captured_text)
    emit(f"=== EPISODE END: survived {steps_survived} steps, total_reward={total_reward:.2f} ===")
    results.append((profile_name, steps_survived, total_reward))
    env.close()

emit()
emit("=== SUMMARY ===")
for i, (name, steps, reward) in enumerate(results, start=1):
    label = f"Episode {i} ({name}):"
    emit(f"{label:<20} steps={steps:<3}  reward={reward:.2f}")

with open(OUTPUT_FILE, "w") as f:
    f.write(buffer.getvalue())

print("Saved ✅")
from IPython.display import FileLink
FileLink('demo_output.txt')

In [ ]:
filepath = "/kaggle/working/worksonmymachine/agent/diagnostic_agent.py"

with open(filepath, 'r') as f:
    content = f.read()

# Fix 1: increase max_new_tokens to 256
content = content.replace(
    "max_new_tokens=150,",
    "max_new_tokens=256,"
)

# Fix 2: remove "Think carefully" and replace with direct instruction
old_prompt_end = """Think carefully before answering. Output EXACTLY in this format:
FAULT: [thermal/memory/power/latchup/none]
SEVERITY: [1/2/3]
CONFIDENCE: [low/medium/high]
REASONING: [one sentence]
"""

new_prompt_end = """Output EXACTLY in this format with no preamble:
FAULT: [thermal/memory/power/latchup/none]
SEVERITY: [1/2/3]
CONFIDENCE: [low/medium/high]
REASONING: [one sentence]
"""

content = content.replace(old_prompt_end, new_prompt_end)

with open(filepath, 'w') as f:
    f.write(content)

print("Fixed ✅")
print("Verifying changes:")
with open(filepath, 'r') as f:
    new = f.read()
print("max_new_tokens=256" in new, "— max tokens fixed")
print("no preamble" in new, "— prompt fixed")
print("Think carefully" not in new, "— think removed")

In [ ]:
# Cell 7 — Demo FAST (fixed diagnostic agent)
import os, sys, torch
from contextlib import redirect_stdout
from io import StringIO

sys.path.insert(0, "/kaggle/working/worksonmymachine")

# reload the fixed agent
import importlib
import agent.diagnostic_agent
importlib.reload(agent.diagnostic_agent)
from agent.diagnostic_agent import DiagnosticAgent
from agent.recovery_agent import RecoveryAgent
from agent.memory import Memory
from agent.run_episode import run_episode
from TITAN_env.core.environment.fault_injection import INTENSITY_PROFILES, FaultInjector
from TITAN_env.core.environment.gym_env import TITANGymEnv

OUTPUT_FILE = "/kaggle/working/demo_output.txt"

diagnostic_agent = DiagnosticAgent(model, tokenizer)
recovery_agent = RecoveryAgent(model, tokenizer)

profile_names = ["low", "medium", "high"]
results = []
buffer = StringIO()

def emit(line=""):
    print(line)
    print(line, file=buffer)

emit("=" * 60)
emit("TITAN Multi-Agent Fault Recovery Demo")
emit("=" * 60)

for episode_idx, profile_name in enumerate(profile_names, start=1):
    emit()
    emit(f"=== EPISODE {episode_idx} — {profile_name} radiation ===")
    env = TITANGymEnv(
        fault_injector=FaultInjector(INTENSITY_PROFILES[profile_name]),
        reward_version="v3",
        training_mode=False,
    )
    memory = Memory(max_size=5)
    captured = StringIO()
    with redirect_stdout(captured):
        total_reward, steps_survived, _ = run_episode(
            env, diagnostic_agent, recovery_agent, memory, max_steps=10
        )
    captured_text = captured.getvalue()
    sys.stdout.write(captured_text)
    buffer.write(captured_text)
    emit(f"=== EPISODE END: survived {steps_survived} steps, total_reward={total_reward:.2f} ===")
    results.append((profile_name, steps_survived, total_reward))
    env.close()

emit()
emit("=== SUMMARY ===")
for i, (name, steps, reward) in enumerate(results, start=1):
    label = f"Episode {i} ({name}):"
    emit(f"{label:<20} steps={steps:<3}  reward={reward:.2f}")

with open(OUTPUT_FILE, "w") as f:
    f.write(buffer.getvalue())

print("Saved ✅")
from IPython.display import FileLink
FileLink('demo_output.txt')
